# Redesign Fase 3: Recursive Forecasting (CEDA-Driven)
Rancangan ulang berdasarkan temuan *Confirmatory EDA* (CEDA).

Strategi Utama:
1. **Interpolasi Imputasi Berbeda**: Linear untuk cuaca lokal dinamis, Forward Fill untuk indeks makro global.
2. **Feature Interactions**: Integrasi curah hujan dengan saturasi tanah (`rainfall` * `soil moisture`) sebagai penanda debit limpasan.
3. **Multi-Model Recursive Blending**: LightGBM dan XGBoost bekerja bahu-membahu dalam siklus tebak-dan-suntik (Predict & Inject) untuk masa depan.
4. Kode *Ultra-Clean* (Murni Eksekusi Tanpa Komentar).

In [1]:
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 150)

## 1. Load Data & Merging (Train-Test Setup)
Penggabungan data primer menjadi matriks kontinu berdasar waktu. Data *test* diawali dengan nilai target `NaN`.

In [2]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
env_data = pd.read_csv('../data/raw/data_pendukung/data_lingkungan.csv')
coords = pd.read_csv('../data/raw/data_pendukung/koordinat_pos.csv')

train['datetime'] = pd.to_datetime(train['datetime'])
env_data['datetime'] = pd.to_datetime(env_data['datetime'])
test['datetime'] = pd.to_datetime(test['id'].str[:19])
test['nama_pos'] = test['id'].str[22:]
test['tma_mdpl'] = np.nan

all_data = pd.concat([train, test], ignore_index=True)
all_data = all_data.sort_values(by=['nama_pos', 'datetime']).reset_index(drop=True)

## 2. Advanced Imputation & Aggregation
Penanganan nilai kosong (Missing Values) yang adaptif:
- Kolom makro iklim global diatasi dengan `ffill`.
- Kolom atmosfer dan kelembapan diatasi dengan `interpolate(linear)`.
Setelah beres, data diaggregasi dari jam ke resolusi 3-Jam.

In [3]:
macro_cols = ['nino_34', 'mjo_phase', 'mjo_amplitude', 'mjo_active', 'rmm1', 'rmm2']
dynamic_cols = ['surface_pressure_hpa', 'pressure_msl_hpa', 'soil_moisture_0_7cm', 
                'soil_moisture_7_28cm', 'soil_moisture_28_100cm', 'soil_moisture_100_255cm']

env_data = env_data.sort_values(['nama_pos', 'datetime'])

for c in macro_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].ffill().bfill()
for c in dynamic_cols:
    env_data[c] = env_data.groupby('nama_pos')[c].apply(lambda x: x.interpolate(method='linear').bfill().ffill()).reset_index(level=0, drop=True)

def aggregate_env_data(df):
    cat_cols = ['nama_pos', 'landcover_name', 'datetime']
    num_cols = [c for c in df.columns if c not in cat_cols]
    
    agg_funcs = {col: 'mean' for col in num_cols}
    agg_funcs['rainfall_mm'] = 'sum'
    agg_funcs['rainfall_openmeteo_mm'] = 'sum'
    agg_funcs['rainfall_max_24h_mm'] = 'max'
    
    df_indexed = df.set_index('datetime')
    agg_df = df_indexed.groupby(['nama_pos', pd.Grouper(freq='3h', label='right', closed='right')]).agg(agg_funcs).reset_index()
    return agg_df

env_agg = aggregate_env_data(env_data)

all_data = pd.merge(all_data, env_agg, on=['datetime', 'nama_pos'], how='left')
all_data = pd.merge(all_data, coords, on='nama_pos', how='left')

## 3. Dinamic Feature Engineering (Interactions & Lags)
Selain *lag* TMA, dibangun pula pemicu interaksi fisik antara curah hujan dan kadar kejenuhan tanah (`runoff_factor`). Fitur waktu (Siklus) diekstraksi ke *Sinus-Cosinus*.

In [4]:
def create_features(df):
    df_temp = df.copy()
    
    df_temp['month'] = df_temp['datetime'].dt.month
    df_temp['hour'] = df_temp['datetime'].dt.hour
    df_temp['sin_hour'] = np.sin(2 * np.pi * df_temp['hour'] / 24)
    df_temp['cos_hour'] = np.cos(2 * np.pi * df_temp['hour'] / 24)
    
    df_temp['runoff_factor'] = df_temp['rainfall_mm'] * df_temp['soil_moisture_0_7cm']
    
    df_temp['tma_lag_1'] = df_temp.groupby('nama_pos')['tma_mdpl'].shift(1)
    df_temp['tma_lag_2'] = df_temp.groupby('nama_pos')['tma_mdpl'].shift(2)
    df_temp['tma_lag_8'] = df_temp.groupby('nama_pos')['tma_mdpl'].shift(8)
    
    df_temp['rainfall_rolling_12h'] = df_temp.groupby('nama_pos')['rainfall_mm'].transform(lambda x: x.rolling(window=4, min_periods=1).sum())
    
    return df_temp

le = LabelEncoder()
all_data['nama_pos_encoded'] = le.fit_transform(all_data['nama_pos'])

## 4. Multi-Model Training (LGBM + XGBoost)
Kedua algoritma raksasa dilatih secara global di data historis. Keistimewaan arsitektur ganda ini adalah memangkas resiko deviasi tajam di masa rekursi.

In [5]:
train_mask = all_data['tma_mdpl'].notnull()
train_data = create_features(all_data[train_mask])

drop_cols = ['datetime', 'nama_pos', 'tma_mdpl', 'id', 'landcover_name']
features = [c for c in train_data.columns if c not in drop_cols]
target = 'tma_mdpl'

train_data = train_data.sort_values('datetime')
split_idx = int(len(train_data) * 0.8)

X_train, y_train = train_data.iloc[:split_idx][features], train_data.iloc[:split_idx][target]
X_val, y_val = train_data.iloc[split_idx:][features], train_data.iloc[split_idx:][target]

lgb_model = lgb.LGBMRegressor(learning_rate=0.03, num_leaves=63, max_depth=8, n_estimators=600, random_state=42, verbose=-1)
xgb_model = xgb.XGBRegressor(learning_rate=0.03, max_depth=7, n_estimators=600, random_state=42)

lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(stopping_rounds=50)])
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

lgb_preds = lgb_model.predict(X_val)
xgb_preds = xgb_model.predict(X_val)
blend_preds = (lgb_preds * 0.6) + (xgb_preds * 0.4)

print("Static Validation RMSE (Optimistic Blend):", np.sqrt(mean_squared_error(y_val, blend_preds)))

lgb_full = lgb.LGBMRegressor(learning_rate=0.03, num_leaves=63, max_depth=8, n_estimators=600, random_state=42, verbose=-1)
xgb_full = xgb.XGBRegressor(learning_rate=0.03, max_depth=7, n_estimators=600, random_state=42)

lgb_full.fit(train_data[features], train_data[target])
xgb_full.fit(train_data[features], train_data[target])

Training until validation scores don't improve for 50 rounds


Early stopping, best iteration is:
[202]	valid_0's l2: 1.04499


Static Validation RMSE (Optimistic Blend): 2.3697813947700013


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.03, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=7, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=600, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

## 5. CEDA Multi-Recursive Inference
Memasuki zona inferensi kronologis absolut. Loop menembus *test set* dengan memanggil `predict` pada kedua model sekaligus, meratakan tebakan (60% LGBM, 40% XGBoost), dan menjejali *dataframe* agar fitur Lag berikutnya membumi (*grounded*).

In [6]:
test_dates = test['datetime'].sort_values().unique()

for current_dt in test_dates:
    updated_all_data = create_features(all_data)
    
    current_mask = (updated_all_data['datetime'] == current_dt) & (updated_all_data['tma_mdpl'].isnull())
    if not current_mask.any():
        continue
        
    X_current = updated_all_data.loc[current_mask, features]
    
    p_lgb = lgb_full.predict(X_current)
    p_xgb = xgb_full.predict(X_current)
    
    current_preds = (p_lgb * 0.6) + (p_xgb * 0.4)
    
    all_data.loc[current_mask, 'tma_mdpl'] = current_preds

print("CEDA Multi-Recursive Inference Tuntas!")

CEDA Multi-Recursive Inference Tuntas!


## 6. Output Final (Submisi)
Menyuling kembali TMA khusus data *test* berdasarkan indeks asli, kemudian mengunci file di rute destinasi.

In [7]:
test_final = all_data[all_data['id'].notnull()].copy()
test_final = test_final[['id', 'tma_mdpl']]

if not os.path.exists('../submissions'):
    os.makedirs('../submissions')

test_final.to_csv('../submissions/submission.csv', index=False)
print("File CEDA-Driven (submission.csv) berhasil disusun.")

File CEDA-Driven (submission.csv) berhasil disusun.
